# Application of Radial Equilibrium Equation for a Rotor - Inverse Problem
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)

The radial equilibrium equation can be reordered and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = 2 \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

The necessary modules are imported

In [18]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 3)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The data for the fan is input in this cell.
The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are computed, and the swirl distribution is also chosen.
For numerical computations, the $r$ domain is divided in $m$ intervals, and $n$ for output of results. Velocity at $r_{rms}$, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$, is calculated from Euler's equation, and $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.

In [19]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 70                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.6837 m/s
cx_rms = 10.3916 m/s


In the next cell the type of the angular flow is chosen. These are the options:

- **Free Vortex**:

  $$ c_\theta = \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
- **Forced Vortex**:
  
  $$ c_\theta = Ar $$
  where
  $$ A = \frac{c_{\theta,rms}}{r_{rms}} $$
- **Constant Vortex**:
  
  $$ c_\theta = c_{\theta,rms} $$
- **Mixed Vortex**:
  
  $$ c_\theta = A(r-r_{rms}) + \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
  and
  $$ A = \frac{\Delta c_\theta}{r_t - r_h} + \frac{B}{r_t r_h} $$
  where $\Delta c_\theta$ is the variability of $c_\theta$ between the tip and the hub, that is, some kind of "strength" of the vortex
- **Arbitray Vortex**:
  
  The user can define any function $c_\theta(r)$
- **Table**:
  
  Tangential velocity can be given as an array, along with the $r_i$ points


In [20]:
# User choice: "free", "forced", "constant", "mixed", "arbitrary", "table"
flow_type = "table"

if flow_type == "free":
    B = ctrms*rrms
    ctheta = B/r
    print("Free vortex flow")
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "forced":
    A = ctrms/(rrms)
    ctheta = A*r
    print("Forced vortex flow")
    print("A = {:.4f} 1/s".format(A))
elif flow_type == "constant":
    ctheta = ctrms*np.ones(r.size)
    print("Constant ctheta flow")
    print("ctheta = {:.4f} m/s".format(ctrms))
elif flow_type == "mixed":
    delta_ctheta = 1
    B = ctrms*rrms
    A = delta_ctheta/(rt-rh) + B/(rt*rh)
    ctheta = A*(r-rrms) + B/r
    print("Mixed vortex flow")
    print("A = {:.4f} 1/s".format(A))
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "arbitrary":
    ctheta = 2*omega/3*r-9.08/np.sqrt(r)
elif flow_type == "table":
    # For example, values obtained from a reference design, CFD, or experimental extraction
    c_theta_array = np.array([0.90278909, 1.28974431, 1.66424824, 2.03205183, 2.39580352,
       2.75690324, 3.11616483, 3.47409568, 3.83102958, 4.1871957 ])
    r_stations = np.linspace(rh, rt, len(c_theta_array))
    ctheta_spline = CubicSpline(r_stations, c_theta_array)
    ctheta = ctheta_spline(r)

And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [21]:
f = 2.0 * cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)


### First approximation

The first approximation of the value of $k$ is made with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [31]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 6.2765 m²/s²


Values of $DF < 0.6$ and a first estimation of $C_D$ are defined. With this assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the geometry of the blade.


$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

The contribution of Samuel Limonchi (course 2023-24 of MUREM) to this part of the notebook is acknowledged.

In [29]:
DF_target = 0.3
CD = 0.05
nu = 1.5e-5 # Kinematic viscosity of air, m²/s
Nblades = 5
def solve_fan(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    alpha2 = np.arctan(np.divide(ctheta,cx))
    beta2 = np.arctan(np.divide(omega*r-ctheta,cx))
    beta1 = np.arctan(np.divide(omega*r,cx))
    cosbeta1 = np.cos(beta1)
    cosbeta2 = np.cos(beta2)
    tanbeta1 = np.tan(beta1)
    tanbeta2 = np.tan(beta2)
    tanbetam = 0.5*(tanbeta1 + tanbeta2)
    beta_m = np.arctan(tanbetam)
    cosbetam = 1 / np.sqrt(1 + tanbetam*tanbetam)
    solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF_target - 2 * (1 - (cosbeta1/cosbeta2) ))
    bladespace = 2*np.pi*r/Nblades
    chord =  solidity * bladespace
    CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
    Delta_p0 = rho * omega * r * ctheta
    Delta_p0_avg = 2*trapezoid(np.multiply(r,Delta_p0),r)/(rt*rt-rh*rh)
    ## Computation of total pressure loss and net total pressure rise
    Wm = cx / cosbetam
    Delta_p0_loss = 0.5 * rho * Wm * Wm * solidity * CD / cosbetam
    Delta_p0_net = Delta_p0 - Delta_p0_loss
    efficiency = Delta_p0_net / Delta_p0
    Delta_p0_avg_net = 2*trapezoid(np.multiply(r,Delta_p0_net),r)/(rt*rt-rh*rh)
    efficiency_avg = Delta_p0_avg_net / Delta_p0_avg
    Re = np.divide(np.multiply(Wm, chord), nu)
    #
    print("Q = {:.3f} m^3/s".format(Q))
    print("Delta p0 avg = {:.2f} Pa".format(Delta_p0_avg))
    print("Delta p0 avg net = {:.2f} Pa".format(Delta_p0_avg_net))
    print("efficiency avg = {:.2f} %".format(efficiency_avg*100))
    errorQ = np.abs(Qdata-Q)/Qdata * 100
    print("error in flow rate = {:.1e} %".format(errorQ))
    errorP0 = np.abs(Delta_p0_target-Delta_p0_avg_net)/Delta_p0_target * 100
    print("error in pressure rise = {:.1e} %".format(errorP0))
    #
    # Results are interpolated to a smaller number of points for output
    #
    rdata = np.linspace(rh, rt, n)
    ctheta_data = CubicSpline(r,ctheta)(rdata)
    cx_data = CubicSpline(r,cx)(rdata)
    alpha2_data = np.rad2deg(CubicSpline(r,alpha2)(rdata))
    beta2_data = np.rad2deg(CubicSpline(r,beta2)(rdata))
    beta1_data = np.rad2deg(CubicSpline(r,beta1)(rdata))
    beta_m_data = np.rad2deg(CubicSpline(r,beta_m)(rdata))
    solidity_data = CubicSpline(r,solidity)(rdata)
    Wm_data = CubicSpline(r,Wm)(rdata)
    CL_data = CubicSpline(r,CL)(rdata)
    Delta_p0_data = CubicSpline(r,Delta_p0)(rdata)
    Delta_p0_loss_data = CubicSpline(r,Delta_p0_loss)(rdata)
    Delta_p0_net_data = CubicSpline(r,Delta_p0_net)(rdata)
    efficiency_data = CubicSpline(r,efficiency)(rdata)
    bladespace_data = 2*np.pi*rdata/Nblades
    chord_data =  solidity_data * bladespace_data
    Re_data = CubicSpline(r,Re)(rdata)
    #
    # Pandas DataFrame is created for output
    #
    df = pd.DataFrame({
        "Radius (m)": rdata,
        "c_theta (m/s)": ctheta_data,
        "c_x (m/s)": cx_data,
        "alpha_2 (deg)": alpha2_data,
        "beta_1 (deg)": beta1_data,
        "beta_2 (deg)": beta2_data,
        "beta_m (deg)": beta_m_data,
        "Solidity": solidity_data,
        "CL": CL_data,
        "Chord (mm)": chord_data * 1000,
        "Wm (m/s)": Wm_data,
        "Delta p0_Euler (Pa)": Delta_p0_data,
        "Delta_p0_loss (Pa)": Delta_p0_loss_data,
        "Delta_p0_net (Pa)": Delta_p0_net_data,
        "efficiency": efficiency_data,
        "Reynolds number": Re_data
    })
    df = df.round({"Chord (mm)":0, "Reynolds number":0})
    return df

In [30]:

df = solve_fan(k)

Q = 0.477 m^3/s
Delta p0 avg = 78.82 Pa
Delta p0 avg net = 67.87 Pa
efficiency avg = 86.10 %
error in flow rate = 1.2e-14 %
error in pressure rise = 3.0e+00 %


In [25]:
df

,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Wm (m/s),Delta p0_Euler (Pa),Delta_p0_loss (Pa),Delta_p0_net (Pa),efficiency,Reynolds number
0,0.032,0.903,2.505,19.817,71.808,69.556,70.745,0.290,0.676,12.0,7.597,8.259,1.524,6.735,0.815,5911.0
1,0.042,1.290,4.124,17.366,67.760,64.879,66.402,0.309,0.696,16.0,10.302,15.609,2.457,13.152,0.843,11288.0
2,0.053,1.664,5.541,16.719,66.174,63.018,64.688,0.319,0.700,21.0,12.959,25.058,3.756,21.302,0.850,18223.0
3,0.063,2.032,6.881,16.452,65.369,62.064,63.813,0.325,0.701,26.0,15.593,36.599,5.365,31.234,0.853,26710.0
4,0.073,2.396,8.184,16.318,64.901,61.504,63.302,0.328,0.702,30.0,18.215,50.228,7.274,42.954,0.855,36743.0
5,0.084,2.757,9.463,16.242,64.603,61.146,62.977,0.331,0.702,35.0,20.828,65.942,9.481,56.462,0.856,48320.0
6,0.094,3.116,10.729,16.196,64.402,60.903,62.756,0.333,0.702,39.0,23.436,83.741,11.983,71.759,0.857,61440.0
7,0.104,3.474,11.984,16.167,64.260,60.731,62.600,0.334,0.702,44.0,26.041,103.623,14.780,88.843,0.857,76100.0
8,0.115,3.831,13.232,16.147,64.155,60.603,62.484,0.335,0.701,48.0,28.642,125.587,17.871,107.716,0.858,92300.0
9,0.125,4.187,14.476,16.133,64.076,60.507,62.397,0.336,0.701,53.0,31.242,149.632,21.256,128.376,0.858,110039.0


### More precise computation


Instead of estimating $k$ with the assumption of $c_x$ in $r_{rms}$ being the average value, a more accurate computation
can be performed forcing the flow rate to be the input one (equation (5.47) and figure 5.6)

In [32]:
from scipy.optimize import brentq

def QFunction(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

k = brentq(QFunction,min(0.5*k,5*k),max(0.5*k,5*k))
print("k = {:.4f} m²/s²".format(k))

k = 15.0563 m²/s²


In [33]:
df = solve_fan(k)
df

Q = 0.477 m^3/s
Delta p0 avg = 78.82 Pa
Delta p0 avg net = 67.87 Pa
efficiency avg = 86.10 %
error in flow rate = 1.2e-14 %
error in pressure rise = 3.0e+00 %


,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Wm (m/s),Delta p0_Euler (Pa),Delta_p0_loss (Pa),Delta_p0_net (Pa),efficiency,Reynolds number
0,0.032,0.903,3.880,13.098,63.025,60.0,61.586,0.255,0.777,10.0,8.155,8.259,1.068,7.191,0.871,5567.0
1,0.042,1.290,5.078,14.251,63.274,60.0,61.724,0.286,0.747,15.0,10.720,15.609,2.084,13.525,0.867,10886.0
2,0.053,1.664,6.283,14.835,63.400,60.0,61.794,0.303,0.732,20.0,13.294,25.058,3.402,21.656,0.864,17788.0
3,0.063,2.032,7.492,15.175,63.472,60.0,61.834,0.313,0.724,25.0,15.872,36.599,5.018,31.581,0.863,26255.0
4,0.073,2.396,8.704,15.391,63.519,60.0,61.860,0.320,0.718,29.0,18.454,50.228,6.931,43.297,0.862,36275.0
5,0.084,2.757,9.916,15.537,63.550,60.0,61.877,0.324,0.714,34.0,21.038,65.942,9.139,56.803,0.861,47842.0
6,0.094,3.116,11.130,15.641,63.572,60.0,61.890,0.328,0.712,39.0,23.623,83.741,11.642,72.099,0.861,60955.0
7,0.104,3.474,12.345,15.718,63.588,60.0,61.899,0.330,0.710,43.0,26.209,103.623,14.440,89.183,0.861,75610.0
8,0.115,3.831,13.560,15.776,63.601,60.0,61.906,0.332,0.708,48.0,28.795,125.587,17.531,108.056,0.860,91806.0
9,0.125,4.187,14.776,15.822,63.611,60.0,61.912,0.333,0.707,52.0,31.382,149.632,20.916,128.716,0.860,109542.0


In [28]:
df['beta_2 (deg)'].to_numpy()

array([60.00012735, 59.99997498, 59.9999888 , 59.99998383, 59.99998533,
       59.99998647, 59.99998784, 59.99998909, 59.99999029, 59.99999117])